# [GIS] SGIS 2024 결합 마스터 생성 — 2026-10-26

SK 집계구 집계 결과에 SGIS 2024를 결합하고 분석용 변수를 선택하는 단계다.
2026-10-14에 실행한 병합·변수 선택 노트북의 계산과 기본 선택 조건을 유지하고,
입출력 경로와 실행 환경을 현재 저장소에 맞췄다.

- 입력: 2025년 7~12월 SK 집계구 통합 CSV, SGIS 2024 원본 CSV 38개, 제공용 코드표
- 병합: `TOT_OA_CD` 14자리 문자열 기준 왼쪽 결합. 월별 원래 행·열·셀 내용을 보존
- 결측: 원본 `N/A`와 원본 행 없음(`no_record`)을 구분하고 0으로 대체하지 않음
- 기본 결과: 집계구 1,698개 × 6개월 = 10,188행, 병합 491열 → 선택 289열 → 업종별 사업체 수 제외 203열
- 경로·의존성·재현 명령: [실행 안내](../docs/sgis_master_reproduction.md)

전체 셀을 순서대로 실행한다. 원본은 읽기만 하고, 같은 이름의 생성 산출물은 다시 저장한다.
공간 경계 동일성을 검증하거나 개인의 고립 여부를 판정하는 단계가 아니다.

## 1. 실행 환경

저장소 루트에서 `uv sync --locked --extra dev`를 실행하고 해당 환경의 Python 커널을 쓴다.
`pandas`와 `.xls` 코드표용 `xlrd`는 기본 의존성, Jupyter는 dev 의존성이다.
노트북에서 패키지를 별도 설치하거나 pandas 버전을 바꾸지 않는다.

In [ ]:
from pathlib import Path
import json
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)

## 2. 저장소 기준 경로

저장소 루트 또는 `notebooks/`에서 실행할 수 있다. 이전 GIS 노트북의 마지막 통합 셀에서
생성한 `data/processed/sk_통합_집계구/sk_통신인구_전체_집계구.csv`를 입력으로 쓴다.
SGIS 원본은 `data/raw/reference_data/sgis2024/`의 인구·가구·주택·사업체 하위 폴더에,
코드표는 `data/raw/reference_data/ref_code/`에 배치한다. 구조가 다르면 아래 입력 경로만 바꾼다.

병합 CSV 3개는 `data/interim/1차 merge/`, 최종 203열 마스터는 `data/processed/`에 저장한다.
선택 목록·보고서와 중간 선택 결과는 `data/processed/sgis_master_validation/`에 저장한다.

In [ ]:
SGIS_YEAR = "2024"
MASTER_YEAR = "2025"


def find_repo_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("저장소 루트 또는 notebooks 폴더에서 실행하세요.")


def resolve_korean_path(path):
    # 한글 NFC/NFD 파일명 차이를 허용하되, 중복 후보는 임의로 고르지 않는다.
    path = Path(path).absolute()
    if path.exists():
        return path
    current = Path(path.anchor)
    for part in path.parts[1:]:
        if not current.is_dir():
            raise FileNotFoundError(f"입력 폴더가 없습니다: {current}")
        matches = [
            p for p in current.iterdir()
            if unicodedata.normalize("NFC", p.name) == unicodedata.normalize("NFC", part)
        ]
        if len(matches) != 1:
            raise FileNotFoundError(f"입력 경로를 찾을 수 없거나 모호합니다: {path}")
        current = matches[0]
    return current


REPO_ROOT = find_repo_root()
MASTER_PATH = REPO_ROOT / "data/processed/sk_통합_집계구/sk_통신인구_전체_집계구.csv"
CODEBOOK_PATH = REPO_ROOT / "data/raw/reference_data/ref_code/3. 제공용 코드(statistics_code).xls"
SGIS_ROOT = REPO_ROOT / "data/raw/reference_data/sgis2024"
SGIS_DIRS = {
    "인구": SGIS_ROOT / "population",
    "가구": SGIS_ROOT / "household",
    "주택": SGIS_ROOT / "housing",
    "사업체·종사자": SGIS_ROOT / "business",
}
MERGE_OUTPUT_DIR = REPO_ROOT / "data/interim/1차 merge"
PROCESSED_DIR = REPO_ROOT / "data/processed"
VALIDATION_DIR = PROCESSED_DIR / "sgis_master_validation"


def relative_path(path):
    return unicodedata.normalize("NFC", Path(path).relative_to(REPO_ROOT).as_posix())


MASTER_PATH = resolve_korean_path(MASTER_PATH)
CODEBOOK_PATH = resolve_korean_path(CODEBOOK_PATH)
SGIS_DIRS = {name: resolve_korean_path(p) for name, p in SGIS_DIRS.items()}
print("마스터:", relative_path(MASTER_PATH))
print("병합 결과:", relative_path(MERGE_OUTPUT_DIR))
print("변수 선택 결과:", relative_path(PROCESSED_DIR))

## 3. 원본 읽기와 중복 검사
마스터는 기존 셀 표기를 유지하도록 문자열로 읽습니다. 새로 추가할 SGIS 값은 숫자로 변환합니다.
SGIS CSV에는 헤더가 없으므로 첫 번째 데이터 행부터 모두 읽습니다.


In [ ]:
master = pd.read_csv(MASTER_PATH, dtype=str, encoding='utf-8-sig', keep_default_na=False)
required = {'TOT_OA_CD', 'STD_YM'}
if not required.issubset(master.columns):
    raise ValueError(f'마스터 필수 열 누락: {required - set(master.columns)}')
assert master['TOT_OA_CD'].str.fullmatch(r'[0-9]{14}').all(), '마스터 코드 형식 오류'
assert master['STD_YM'].str.fullmatch(r'[0-9]{6}').all(), '마스터 기준월 형식 오류'
assert master['STD_YM'].str[:4].eq(MASTER_YEAR).all(), '설정한 마스터 연도와 다릅니다.'
if master.duplicated(['STD_YM', 'TOT_OA_CD']).any():
    raise ValueError('마스터에 기준월 + 집계구 코드 중복이 있습니다.')

parts, manifest = [], []
for category, directory in SGIS_DIRS.items():
    paths = sorted(directory.glob('*.csv'))
    if not paths:
        raise FileNotFoundError(f'CSV가 없습니다: {directory}')
    for path in paths:
        raw = pd.read_csv(path, header=None, dtype=str,
                          encoding='utf-8-sig', keep_default_na=False)
        if raw.shape[1] != 4 or raw.empty:
            raise ValueError(f'4열 데이터 형식이 아닙니다: {path}')
        raw.columns = ['SGIS_YEAR', 'TOT_OA_CD', 'VARIABLE_CODE', 'VALUE_RAW']
        raw['CATEGORY'] = category
        raw['SOURCE_FILE'] = unicodedata.normalize('NFC', path.name)
        parts.append(raw)
        manifest.append({'category': category, 'path': relative_path(path), 'rows': len(raw)})

sgis = pd.concat(parts, ignore_index=True)
assert sgis['SGIS_YEAR'].eq(SGIS_YEAR).all(), 'SGIS에 다른 연도 자료가 섞여 있습니다.'
assert sgis['TOT_OA_CD'].str.fullmatch(r'[0-9]{14}').all(), 'SGIS 코드 형식 오류'
assert sgis['VARIABLE_CODE'].str.fullmatch(r'[A-Za-z0-9_]+').all(), '변수 코드 형식 오류'
keys = ['SGIS_YEAR', 'TOT_OA_CD', 'VARIABLE_CODE']
duplicates = sgis.duplicated(keys, keep=False)
if duplicates.any():
    display(sgis.loc[duplicates].head(20))
    raise ValueError('SGIS 키가 중복됩니다. 합계나 첫 값으로 처리하지 않고 중단합니다.')

is_source_na = sgis['VALUE_RAW'].eq('N/A')
sgis['VALUE'] = pd.to_numeric(sgis['VALUE_RAW'].mask(is_source_na), errors='coerce')
unexpected = sgis['VALUE'].isna() & ~is_source_na
if unexpected.any():
    display(sgis.loc[unexpected, ['SOURCE_FILE', 'VARIABLE_CODE', 'VALUE_RAW']].head(20))
    raise ValueError('N/A 외의 숫자가 아닌 값이 있습니다. 원본 의미를 확인하세요.')
sgis['VALUE_STATUS'] = 'numeric'
sgis.loc[is_source_na, 'VALUE_STATUS'] = 'source_na'

print(f'마스터: {master.shape[0]:,}행 × {master.shape[1]:,}열')
print(f'SGIS: {len(manifest)}개 파일, {len(sgis):,}행, 변수 {sgis.VARIABLE_CODE.nunique()}개')
display(sgis.groupby('CATEGORY').agg(원본행수=('VALUE_RAW', 'size'),
                                    변수수=('VARIABLE_CODE', 'nunique')))

## 4. 코드표 연결
`집계구·행정동` 시트에서 실제 CSV에 등장하는 코드만 가져옵니다.
산업분류는 **11차**를 선택합니다. 동일 코드가 8·9·10차에도 재사용되므로 코드만으로 이름을 고르면 안 됩니다.
건축년도는 최신 구간을 사용합니다. 해당 구간 제목은 `2015년~2023년`이나 목록에는
`ho_yr_020 = 2024년`까지 있으므로, 코드표 원문을 보존하고 확인 메모를 남깁니다.
변수 설명표에는 원본 파일명과 마스터 집계구 기준 결측 현황도 포함합니다.


In [ ]:
codebook = pd.read_excel(CODEBOOK_PATH, sheet_name='집계구·행정동',
                         header=None, dtype=str, engine='xlrd').fillna('')
variables = sorted(sgis['VARIABLE_CODE'].unique())
codebook['CODEBOOK_SECTION'] = codebook[2].replace('', pd.NA).ffill().fillna('')
industrial = codebook[4].str.startswith(('cp_', 'cp2_'))
housing_year = codebook[4].str.startswith('ho_yr_')
appropriate = (~industrial | codebook['CODEBOOK_SECTION'].str.contains('11차', regex=False))
appropriate &= (~housing_year | codebook['CODEBOOK_SECTION'].str.contains('2015년~2023년', regex=False))
dictionary = codebook.loc[codebook[4].isin(variables) & appropriate,
                          [3, 4, 'CODEBOOK_SECTION']].copy()
dictionary.columns = ['VARIABLE_NAME', 'VARIABLE_CODE', 'CODEBOOK_SECTION']
dictionary['MAPPING_NOTE'] = ''
dictionary.loc[dictionary['VARIABLE_CODE'].str.startswith('ho_yr_'), 'MAPPING_NOTE'] = (
    '코드표 최신 건축년도 구간 사용. 구간 제목은 2015년~2023년이나 2024년 항목까지 기재되어 있어 연도 표기 확인 필요.'
)
dictionary = dictionary.drop_duplicates()
if dictionary['VARIABLE_CODE'].duplicated().any():
    raise ValueError('코드표에 같은 코드의 서로 다른 이름이 있습니다.')
unmapped = sorted(set(variables) - set(dictionary['VARIABLE_CODE']))
if unmapped:
    raise ValueError(f'코드표에 없는 변수: {unmapped}')

metadata = sgis.groupby('VARIABLE_CODE').agg(
    CATEGORY=('CATEGORY', lambda s: ' / '.join(sorted(set(s)))),
    SOURCE_FILES=('SOURCE_FILE', lambda s: ' | '.join(sorted(set(s)))),
).reset_index()
dictionary = dictionary.merge(metadata, on='VARIABLE_CODE', validate='one_to_one')
dictionary['OUTPUT_COLUMN'] = 'SGIS' + SGIS_YEAR + '_' + dictionary['VARIABLE_CODE']
dictionary['SGIS_YEAR'] = SGIS_YEAR
dictionary = dictionary.sort_values('VARIABLE_CODE').reset_index(drop=True)
display(dictionary[['OUTPUT_COLUMN', 'VARIABLE_NAME', 'CATEGORY']].head(12))


## 5. 집계구 코드 대조
마스터에만 있는 코드는 병합 후 SGIS 값이 결측이 됩니다. SGIS에만 있는 코드는 마스터에 새 행으로 추가하지 않습니다.
현재 제공 자료는 두 방향 차집합 모두 0개입니다. 불일치가 생기면 아래 결과에서 확인할 수 있습니다.


In [ ]:
master_codes = set(master['TOT_OA_CD'])
sgis_codes = set(sgis['TOT_OA_CD'])
master_only = sorted(master_codes - sgis_codes)
sgis_only = sorted(sgis_codes - master_codes)

print(f'마스터 집계구: {len(master_codes):,}개')
print(f'SGIS 집계구: {len(sgis_codes):,}개')
print(f'일치: {len(master_codes & sgis_codes):,}개')
print(f'마스터에만 존재: {len(master_only):,}개 / SGIS에만 존재: {len(sgis_only):,}개')
if master_only or sgis_only:
    print('마스터에만 있는 코드:', master_only[:20])
    print('SGIS에만 있는 코드:', sgis_only[:20])

match_rows = []
for month, group in master.groupby('STD_YM', sort=True):
    codes = set(group['TOT_OA_CD'])
    match_rows.append({'STD_YM': month, 'MASTER_CODES': len(codes),
                       'MATCHED_CODES': len(codes & sgis_codes),
                       'UNMATCHED_CODES': len(codes - sgis_codes)})
monthly_match = pd.DataFrame(match_rows)
display(monthly_match)


## 6. 피벗과 병합
`pivot_table(..., aggfunc='sum')` 대신 중복을 허용하지 않는 `pivot`을 사용합니다.
`validate='many_to_one'`으로 마스터 행이 불어나는 잘못된 결합을 막습니다.
SGIS는 2024년으로 고정했으므로 병합 키는 집계구 코드 하나입니다.

상태표는 집계구당 한 행이며 값은 `numeric`(숫자), `source_na`(원본 N/A),
`no_record`(원본에 해당 집계구·변수 행 없음)입니다.


In [ ]:
prefix = f'SGIS{SGIS_YEAR}_'
wide = sgis.pivot(index='TOT_OA_CD', columns='VARIABLE_CODE', values='VALUE')
wide = wide.reindex(columns=variables).add_prefix(prefix)
wide.columns.name = None
collisions = set(master.columns) & set(wide.columns)
if collisions:
    raise ValueError(f'이미 같은 SGIS 열이 있습니다: {sorted(collisions)[:10]}')

merged = master.merge(wide.reset_index(), on='TOT_OA_CD', how='left',
                      sort=False, validate='many_to_one')
assert len(merged) == len(master), '병합 후 행 수가 달라졌습니다.'
pd.testing.assert_frame_equal(merged[master.columns], master)
assert merged.shape[1] == master.shape[1] + len(variables)

status_wide = sgis.pivot(index='TOT_OA_CD', columns='VARIABLE_CODE', values='VALUE_STATUS')
status_wide = status_wide.reindex(index=sorted(master_codes), columns=variables).fillna('no_record')
status_wide.columns.name = None
coverage = pd.DataFrame({
    'VARIABLE_CODE': variables,
    'NUMERIC_COUNT': status_wide.eq('numeric').sum().reindex(variables).to_numpy(),
    'SOURCE_NA_COUNT': status_wide.eq('source_na').sum().reindex(variables).to_numpy(),
    'NO_RECORD_COUNT': status_wide.eq('no_record').sum().reindex(variables).to_numpy(),
})
coverage['MISSING_RATE'] = (coverage['SOURCE_NA_COUNT'] + coverage['NO_RECORD_COUNT']) / len(master_codes)
dictionary = dictionary.merge(coverage, on='VARIABLE_CODE', validate='one_to_one')

# 결측 상태와 실제 숫자 여부가 모든 셀에서 일치하는지 확인
aligned_values = wide.reindex(sorted(master_codes))
assert (aligned_values.notna().to_numpy() == status_wide.eq('numeric').to_numpy()).all()
assert merged[list(wide.columns)].notna().to_numpy().sum() == (
    status_wide.eq('numeric').sum(axis=1).reindex(master['TOT_OA_CD']).sum()
)

print(f'병합 완료: {merged.shape[0]:,}행 × {merged.shape[1]:,}열')
print(f'추가 변수: {len(variables)}개 / 기존 행 순서와 셀 내용 보존 확인')
preview_columns = ['STD_YM', 'TOT_OA_CD'] + [prefix + code for code in
    ['to_in_001', 'to_ga_001', 'to_ho_001', 'to_fa_010', 'to_em_020'] if prefix + code in merged]
display(merged[preview_columns].head())


## 7. 새 파일로 저장

1. `master_sgis2024.csv`: 기존 마스터 + SGIS 변수 전체. 결측은 빈 칸으로 저장합니다.
2. `sgis2024_variable_dictionary.csv`: 코드·한글명·출처·변수별 결측 현황.
3. `sgis2024_value_status.csv`: 집계구별 결측 상태. 월별로 반복 저장하지 않습니다.
4. `merge_report.json`: 사용 경로, 연도, 매칭 결과, 원본 행 수, 검증 결과.

CSV는 UTF-8 BOM으로 저장합니다. 분석 시 코드는 문자열로 다시 읽으세요.
SGIS 연간 값은 6개월에 반복되므로 **월을 합칠 때 SGIS 인구·가구수를 더하지 않습니다.**


In [ ]:
OUTPUT_DIR = MERGE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

merged_path = OUTPUT_DIR / f'master_sgis{SGIS_YEAR}.csv'
dictionary_path = OUTPUT_DIR / f'sgis{SGIS_YEAR}_variable_dictionary.csv'
status_path = OUTPUT_DIR / f'sgis{SGIS_YEAR}_value_status.csv'
report_path = VALIDATION_DIR / 'merge_report.json'

merged.to_csv(merged_path, index=False, encoding='utf-8-sig', na_rep='')
dictionary.to_csv(dictionary_path, index=False, encoding='utf-8-sig')
status_wide.add_prefix(prefix).reset_index().to_csv(status_path, index=False, encoding='utf-8-sig')

# 저장된 파일에서도 기존 열의 모든 셀과 행 순서가 보존되었는지 확인
saved_original = pd.read_csv(merged_path, usecols=list(master.columns), dtype=str,
                             encoding='utf-8-sig', keep_default_na=False)
pd.testing.assert_frame_equal(saved_original[master.columns], master)

report = {
    'sgis_year': SGIS_YEAR,
    'master_year': MASTER_YEAR,
    'master_path': relative_path(MASTER_PATH),
    'codebook_path': relative_path(CODEBOOK_PATH),
    'source_files': manifest,
    'master_rows': len(master), 'master_columns': len(master.columns),
    'sgis_rows': len(sgis), 'sgis_variables': len(variables),
    'merged_rows': len(merged), 'merged_columns': len(merged.columns),
    'master_unique_codes': len(master_codes), 'sgis_unique_codes': len(sgis_codes),
    'matched_unique_codes': len(master_codes & sgis_codes),
    'master_only_codes': master_only, 'sgis_only_codes': sgis_only,
    'monthly_match': monthly_match.to_dict(orient='records'),
    'original_na_rows': int(is_source_na.sum()),
    'master_base_dates': sorted(master['BASE_DATE'].unique()) if 'BASE_DATE' in master else [],
    'master_cells_preserved_after_save': True,
    'geographic_boundary_equality_verified': False,
    'missing_status_definitions': {'numeric': '원본 숫자',
                                   'source_na': '원본 N/A', 'no_record': '해당 원본 행 없음'},
    'year_policy': '2024 SGIS 고정 특성을 2025년 모든 월에 연결',
}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
for path in [merged_path, dictionary_path, status_path, report_path]:
    print(path)
print('저장 후 기존 마스터 셀 보존 검증 완료')

## 8. 결과 다시 읽기
저장 후 다른 분석에서 사용하는 예시입니다. 아래 셀은 선택적으로 실행하세요.


In [ ]:
# result = pd.read_csv(
#     merged_path, encoding='utf-8-sig',
#     dtype={'TOT_OA_CD': str, 'STD_YM': str, 'ADM_CD': str, 'BASE_DATE': str},
# )
# result.head()


## 9. 병합 결과에서 분석 변수 선택

앞에서 만든 `merged`와 `dictionary`를 그대로 사용한다. 원본은 수정하지 않고,
선택용 사본의 결측만 빈 문자열로 표현해 셀 내용을 비교한다. 0 대체는 하지 않는다.
기본 선택은 491열에서 202열을 제외한 289열이며, 중복·상수 조건을 현재 자료로 재검사한다.
설정을 바꿀 때는 11번 이후를 다시 실행한다. 최종 셀에서 업종별 사업체 수 86열도 제외한다.

In [ ]:
if not all(name in globals() for name in ['merged', 'dictionary', 'OUTPUT_DIR', 'merged_path', 'dictionary_path']):
    raise RuntimeError('위쪽 병합 및 7번 저장 단계부터 실행하세요.')
selection_source = merged.fillna('').astype(str).copy()
KEYS = ['STD_YM', 'TOT_OA_CD']
if not set(KEYS).issubset(selection_source.columns):
    raise ValueError('STD_YM, TOT_OA_CD 열이 필요합니다.')
if selection_source.duplicated(KEYS).any():
    raise ValueError('기준월 + 집계구 코드가 중복됩니다.')
if dictionary['OUTPUT_COLUMN'].duplicated().any():
    raise ValueError('설명표에 중복 열 이름이 있습니다.')
sgis_columns = [c for c in selection_source if c.startswith('SGIS2024_')]
if not set(sgis_columns).issubset(dictionary['OUTPUT_COLUMN']):
    raise ValueError('설명표에 없는 SGIS 변수가 있습니다.')
if not selection_source.groupby('TOT_OA_CD')[sgis_columns].nunique(dropna=False).le(1).all().all():
    raise ValueError('동일 집계구의 SGIS 값이 월별로 다릅니다. 원본을 확인하세요.')
unique_areas = selection_source.drop_duplicates('TOT_OA_CD')
names = dictionary.set_index('OUTPUT_COLUMN')['VARIABLE_NAME'].to_dict()

print(f'원본: {len(selection_source):,}행 × {len(selection_source.columns):,}열')
print(f'집계구: {len(unique_areas):,}개 / SGIS 변수: {len(sgis_columns)}개')

## 10. 사용할 변수 선택 — 이 셀을 수정하세요

`DROP_GROUPS`의 **True는 해당 그룹 제외**, **False는 유지**입니다.
`KEEP_COLUMNS`에 넣은 열은 그룹에서 제외하더라도 다시 살립니다.
`DROP_COLUMNS`는 개별 제외 목록이며, KEEP과 DROP에 같은 열을 넣으면 오류로 알려줍니다.
열 이름이 틀리면 무시하지 않고 중단합니다.

예시: 중분류 종사자수는 모두 제외하되 소매업 종사자수만 남기려면
`KEEP_COLUMNS = ['SGIS2024_cp2_bem_47']`로 설정하세요.

`CORE_ONLY=False`가 기본입니다. 나중에 핵심 열만 직접 고르고 싶으면 True로 바꾸고
`CORE_COLUMNS`에 전체 열 이름을 나열하세요. 핵심 선택 모드에서도 KEEP 목록은 적용됩니다.
식별·지역·면적 정보는 자동으로 보호되며, 파생지표는 이 단계에서 계산하지 않습니다.

In [ ]:
DROP_GROUPS = {
    'sgis_medium_employees': True,    # cp2_bem_* 71개: 중분류 종사자수
    'sgis_large_employees': True,     # cp_bem_* 19개: 대분류 종사자수
    'sgis_recent_build_year': True,   # ho_yr_006~020 15개: 2010~2024 개별 건축연도
    'sgis_all_missing': True,         # 4세대가구 및 고령 성별 인구 4개: 모두 공백
    'sgis_duplicate_business': True,  # 중분류 사업체수 중 대분류와 동일한 4개
    'sk_duplicate_quality': True,     # 변수별 n_valid/missing_rate 86개
    'sk_zero_partial': True,          # 전부 0인 partial 품질 열 3개
}

KEEP_COLUMNS = []   # 예: ['SGIS2024_cp2_bem_47']
DROP_COLUMNS = []   # 예: ['SGIS2024_ho_ar_001']

CORE_ONLY = False
CORE_COLUMNS = []  # True일 때 반드시 직접 작성: ['SGIS2024_to_in_001', ...]

## 11. 추천 그룹과 선택 결과 계산
중복 열은 남길 열과 실제로 같은지, 전부 공백·0인 열은 현재 자료에서도 그런지 재검사합니다.
새 자료에서 조건이 달라졌으면 중단합니다. 오류가 난 그룹을 False로 바꾸거나 원본을 확인하세요.
총종사자수 등 분석 변수는 강제로 보호하지 않으므로 핵심 선택 모드에서는 원하는 열을 직접 포함해야 합니다.

In [ ]:
sgis_duplicates = {
    'SGIS2024_cp2_bnu_35': 'SGIS2024_cp_bnu_004',
    'SGIS2024_cp2_bnu_68': 'SGIS2024_cp_bnu_012',
    'SGIS2024_cp2_bnu_84': 'SGIS2024_cp_bnu_015',
    'SGIS2024_cp2_bnu_85': 'SGIS2024_cp_bnu_016',
}
sk_duplicates = {}
for c in selection_source.columns:
    if c.endswith('_n_valid'):
        target = 'n_complete'
    elif c.endswith('_missing_rate'):
        target = 'incomplete_rate'
    else:
        continue
    if c.startswith(('MAN_FLOW_POP_CNT_', 'WMAN_FLOW_POP_CNT_')):
        sk_duplicates[c] = 'age_' + target
    elif c.startswith('TMST_'):
        sk_duplicates[c] = 'hour_' + target
    elif c.startswith('FLOW_POP_CNT_'):
        sk_duplicates[c] = 'weekday_' + target

groups = {
    'sgis_medium_employees': [c for c in selection_source if c.startswith('SGIS2024_cp2_bem_')],
    'sgis_large_employees': [c for c in selection_source if c.startswith('SGIS2024_cp_bem_')],
    'sgis_recent_build_year': [f'SGIS2024_ho_yr_{i:03d}' for i in range(6, 21)],
    'sgis_all_missing': ['SGIS2024_ga_sd_004', 'SGIS2024_in_age_050',
                         'SGIS2024_in_age_051', 'SGIS2024_in_age_081'],
    'sgis_duplicate_business': list(sgis_duplicates),
    'sk_duplicate_quality': list(sk_duplicates),
    'sk_zero_partial': ['age_n_partial', 'hour_n_partial', 'weekday_n_partial'],
}
group_reasons = {
    'sgis_medium_employees': '중분류 종사자수: 높은 공백률, 총종사자수 중심 분석',
    'sgis_large_employees': '대분류 종사자수: 고용 업종 구성을 이번 분석에서 제외',
    'sgis_recent_build_year': '개별 건축연도 주택수: 91.6~97.9% 공백(기존 자료 검토값)',
    'sgis_all_missing': '현재 자료 전체 공백',
    'sgis_duplicate_business': '유지한 대분류 사업체수와 값·공백 위치 동일',
    'sk_duplicate_quality': '유지한 분야별 대표 품질 열과 값 동일',
    'sk_zero_partial': '현재 자료 전체 0',
}
PROTECTED = [c for c in ['STD_YM', 'TOT_OA_CD', 'REGION', 'BASE_DATE', 'ADM_CD', 'AREA_M2'] if c in selection_source]

def select_columns():
    if set(DROP_GROUPS) != set(groups):
        raise ValueError('DROP_GROUPS의 그룹 이름을 바꾸지 말고 True/False만 수정하세요.')
    if not all(type(v) is bool for v in DROP_GROUPS.values()):
        raise ValueError('그룹 선택값은 True 또는 False여야 합니다.')
    requested = set(KEEP_COLUMNS + DROP_COLUMNS + CORE_COLUMNS)
    unknown = requested - set(selection_source.columns)
    if unknown:
        raise ValueError(f'없는 열 이름: {sorted(unknown)}')
    if set(KEEP_COLUMNS) & set(DROP_COLUMNS):
        raise ValueError('같은 열을 KEEP_COLUMNS와 DROP_COLUMNS 양쪽에 넣었습니다.')
    if set(PROTECTED) & set(DROP_COLUMNS):
        raise ValueError('식별·지역·면적 보호 열은 제외할 수 없습니다.')
    if CORE_ONLY and not CORE_COLUMNS:
        raise ValueError('CORE_ONLY=True이면 CORE_COLUMNS를 입력하세요.')

    reasons = {}
    if CORE_ONLY:
        selected = set(CORE_COLUMNS) | set(PROTECTED)
        reasons = {c: '핵심 선택 목록에 없음' for c in selection_source if c not in selected}
    else:
        selected = set(selection_source.columns)
        for group, cols in groups.items():
            if DROP_GROUPS[group]:
                absent = set(cols) - set(selection_source.columns)
                if absent:
                    raise ValueError(f'{group}: 예상 열 누락 {sorted(absent)}')
                selected.difference_update(cols)
                reasons.update({c: group_reasons[group] for c in cols})
    selected.update(KEEP_COLUMNS)
    selected.difference_update(DROP_COLUMNS)
    selected.update(PROTECTED)
    reasons.update({c: '사용자 개별 제외' for c in DROP_COLUMNS})

    if not CORE_ONLY:
        for group, mapping in [('sgis_duplicate_business', sgis_duplicates),
                               ('sk_duplicate_quality', sk_duplicates)]:
            if not DROP_GROUPS[group]:
                continue
            for old, replacement in mapping.items():
                if old in selected or old in DROP_COLUMNS:
                    continue
                if replacement not in selected:
                    raise ValueError(f'{old}의 대체 열 {replacement}도 제외되어 있습니다.')
                if not selection_source[old].equals(selection_source[replacement]):
                    raise ValueError(f'현재 자료에서는 중복이 아닙니다: {old}, {replacement}')
        if DROP_GROUPS['sgis_all_missing']:
            for c in groups['sgis_all_missing']:
                if c not in selected and not selection_source[c].isin(['', 'N/A']).all():
                    raise ValueError(f'숫자값이 생긴 열입니다: {c}')
        if DROP_GROUPS['sk_zero_partial']:
            for c in groups['sk_zero_partial']:
                if c not in selected and not pd.to_numeric(selection_source[c], errors='coerce').eq(0).all():
                    raise ValueError(f'전부 0인 열이 아닙니다: {c}')
    ordered = [c for c in selection_source if c in selected]
    excluded = [c for c in selection_source if c not in selected]
    return ordered, excluded, reasons

selected_columns, excluded_columns, reasons = select_columns()
print(f'행 수 유지: {len(selection_source):,}행')
print(f'전체 {len(selection_source.columns)}열 → 유지 {len(selected_columns)}열 / 제외 {len(excluded_columns)}열')
display(pd.DataFrame([
    {'그룹': key, '그룹열수': len(cols), '실제제외': len(set(cols) & set(excluded_columns))}
    for key, cols in groups.items()
]))

## 12. 변수별로 확인하기
`selection_table`에서 유지·제외 여부, 이유, 한글 변수명, 공백률을 확인합니다.
공백률은 저장된 값이 빈 칸 또는 N/A인 비율이며, 원본 N/A와 기록 없음의 원인 구분은 기존 상태표를 참고하세요.
원하는 열을 찾은 다음 10번 셀을 수정하고 11번부터 다시 실행하면 됩니다.

In [ ]:
rows = []
for c in selection_source.columns:
    is_sgis = c in sgis_columns
    values = unique_areas[c] if is_sgis else selection_source[c]
    missing = values.isin(['', 'N/A'])
    rows.append({
        'COLUMN': c, 'NAME': names.get(c, c),
        'SELECTED': c in selected_columns,
        'REASON': ('사용자 개별 유지' if c in KEEP_COLUMNS else '유지') if c in selected_columns else reasons[c],
        'MISSING_RATE': float(missing.mean()),
        'N_OBSERVED': int((~missing).sum()),
        'DENOMINATOR': len(values),
        'RATE_BASIS': '집계구' if is_sgis else '집계구×월',
    })
selection_table = pd.DataFrame(rows)
display(selection_table.loc[~selection_table.SELECTED].head(25))

# 전체 제외 목록을 화면에서 보려면 실행:
# display(selection_table.loc[~selection_table.SELECTED])
# 특정 단어로 찾기:
# display(selection_table[selection_table.NAME.str.contains('1인가구', regex=False)])
# 남기는 SGIS 변수:
# display(selection_table[selection_table.SELECTED & selection_table.COLUMN.str.startswith('SGIS2024_')])

## 13. 선택 결과와 근거 저장

중간 289열 마스터, 변수별 선택 목록, 선택된 변수 설명표와 보고서를
`data/processed/sgis_master_validation/`에 저장한다. 행·열 순서와 유지한 셀 내용을 검증한다.
다음 셀까지 실행하면 실제 후속 분석 입력인 203열 마스터가 `data/processed/`에 생성된다.

In [ ]:
# 설정 변경 후 이전 미리보기와 다른 상태로 저장하지 않도록 확인
current_selected, current_excluded, _ = select_columns()
if current_selected != selected_columns or current_excluded != excluded_columns:
    raise ValueError('설정이 바뀌었습니다. 11번과 12번을 다시 실행한 뒤 저장하세요.')
selected_master = selection_source.loc[:, selected_columns].copy()
pd.testing.assert_frame_equal(selected_master[KEYS], selection_source[KEYS])
pd.testing.assert_frame_equal(selected_master, selection_source[selected_columns])

SELECTION_OUTPUT_DIR = VALIDATION_DIR
SELECTION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result_path = SELECTION_OUTPUT_DIR / 'master_sgis2024_selected.csv'
selected_master.to_csv(result_path, index=False, encoding='utf-8-sig')
selection_table.to_csv(SELECTION_OUTPUT_DIR / 'variable_selection.csv', index=False, encoding='utf-8-sig')
selected_dictionary = dictionary[dictionary.OUTPUT_COLUMN.isin(selected_columns)].copy()
selected_dictionary.to_csv(SELECTION_OUTPUT_DIR / 'sgis2024_variable_dictionary_selected.csv', index=False, encoding='utf-8-sig')
restored = pd.read_csv(result_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
pd.testing.assert_frame_equal(restored, selected_master)

report = {
    'input_master': relative_path(merged_path), 'input_dictionary': relative_path(dictionary_path),
    'rows_before': len(selection_source), 'rows_after': len(selected_master),
    'columns_before': len(selection_source.columns), 'columns_after': len(selected_columns),
    'drop_groups': DROP_GROUPS, 'keep_columns': KEEP_COLUMNS, 'drop_columns': DROP_COLUMNS,
    'core_only': CORE_ONLY, 'core_columns': CORE_COLUMNS,
    'selected_columns': selected_columns, 'excluded_columns': excluded_columns,
    'protected_columns': PROTECTED,
    'retained_cells_and_row_order_preserved': True,
    'missing_values_imputed': False,
}
(SELECTION_OUTPUT_DIR / 'selection_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'저장 완료: {len(selected_master):,}행 × {len(selected_columns)}열')
print('결과 폴더:', SELECTION_OUTPUT_DIR)
print('원본 보존 / 행 순서 및 선택한 셀 내용 보존 검증 완료')

In [ ]:
# 앞서 변수 선택을 마친 selected_master 기준
business_columns = [
    col for col in selected_master.columns
    if col.startswith((
        "SGIS2024_cp2_bnu_",  # 중분류 사업체수
        "SGIS2024_cp_bnu_",   # 대분류 사업체수
    ))
]

selected_master = selected_master.drop(columns=business_columns)

print(f"삭제한 열: {len(business_columns)}개")
print(f"남은 데이터: {selected_master.shape[0]:,}행 × {selected_master.shape[1]}열")

# 후속 코드가 읽는 최종 경로에 생성 산출물을 저장
output_path = PROCESSED_DIR / "master_sgis2024_selected_no_industry_counts.csv"
selected_master.to_csv(output_path, index=False, encoding="utf-8-sig")

# 남은 SGIS 변수 설명표도 함께 저장
dictionary[
    dictionary["OUTPUT_COLUMN"].isin(selected_master.columns)
].to_csv(
    SELECTION_OUTPUT_DIR / "sgis2024_dictionary_no_industry_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장 위치:", output_path)

restored_final = pd.read_csv(output_path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
pd.testing.assert_frame_equal(restored_final, selected_master)
print("최종 파일 행·열 순서와 셀 내용 재로딩 검증 완료")